# Assignment 1 — Gradio dashboard
**Louis de Perier — BBADBA 5B**

Loads the artifacts written by the training stage and serves four tabs.
There is **no training code in this notebook**.

In [ ]:
# Bootstrap: makes this notebook run in Google Colab AND under main.py.
# Pure Python - no ! or % magics, because `jupyter nbconvert --to script`
# turns those into get_ipython() calls that crash outside a kernel.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not Path("features.py").exists():
    REPO = "https://github.com/YOUR-USERNAME/daiist-assignment-1.git"  # <-- EDIT
    subprocess.run(["git", "clone", "-q", REPO, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch", "gradio", "plotly", "scikit-learn"], check=True)
    print("Colab ready, working in", os.getcwd())
else:
    print("Local/CI run, working in", os.getcwd())

In [ ]:
"""Assignment 1 - Gradio dashboard.

Run with:  uv run python main.py app

Reads only the artifacts written by train.py. There is no training code in
this file and nothing is fitted at startup: every model is 168 saved numbers
(one intercept + 167 coefficients) and every prediction is a dot product.
"""

import json
import pickle
from pathlib import Path

import gradio as gr
import numpy as np
import pandas as pd
import plotly.graph_objects as go

ROOT = Path(__file__).resolve().parent
ART = ROOT / "artifacts"

## Load artifacts

These run once at import. Every function below reads these variables — there is no `fit` anywhere in this file, so the app *cannot* retrain.

In [ ]:
# ---------------------------------------------------------------- load artifacts
PARAMS = {k: v for k, v in np.load(ART / "params.npz").items()}
RESULTS = json.loads((ART / "results.json").read_text())
with open(ART / "preprocessor.pkl", "rb") as f:
    PRE = pickle.load(f)
RAW_TEST = pd.read_csv(ART / "raw_test.csv")
RAW_TRAIN = pd.read_csv(ART / "raw_train.csv")

MODELS = ["sklearn", "manual_torch", "standard_torch", "baseline"]
LABELS = {"sklearn": "scikit-learn (closed form)",
          "manual_torch": "Manual PyTorch loop",
          "standard_torch": "Standard nn.Module + optim",
          "baseline": "Naive baseline (train mean)"}
APPRAISAL_COST = 500

# Predictions and truth are already computed and stored in log space.
PRED = {m: {s: np.array(v) for s, v in d.items()}
        for m, d in RESULTS["predictions"].items()}
TRUTH = {s: np.array(v) for s, v in RESULTS["truth"].items()}

PALETTE = {"sklearn": "#2563eb", "manual_torch": "#16a34a",
           "standard_torch": "#d97706", "baseline": "#94a3b8"}

## Helpers

`predict_rows` pushes raw rows through the **fitted** preprocessor loaded from disk, so serving matches training exactly.

In [ ]:
# ----------------------------------------------------------------- helpers

def dollars(log_values):
    """Our models predict log(price); the lender thinks in dollars."""
    return np.exp(np.asarray(log_values))


def predict_rows(model, frame):
    """Score raw rows through the SAME fitted preprocessor used in training."""
    X = PRE.transform(frame.drop(columns=["SalePrice", "Yr Sold"]))
    p = PARAMS[model]
    return p[0] + X @ p[1:]


BLANK = dict(plot_bgcolor="white", paper_bgcolor="white",
             margin=dict(l=60, r=20, t=50, b=50), font=dict(size=13))


def style(fig, title, xlab, ylab):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, **BLANK)
    fig.update_xaxes(gridcolor="#eef1f5", zeroline=False)
    fig.update_yaxes(gridcolor="#eef1f5", zeroline=False)
    return fig

## Tab 1 — Model comparison

Predicted vs actual with a 45° reference line, the metrics table, the coefficient-agreement numbers, and the loss curves descending onto sklearn's exact optimum.

In [ ]:
# ------------------------------------------------------ TAB 1: model comparison

def scatter_pred_actual(selected, split, log_scale):
    fig = go.Figure()
    actual = dollars(TRUTH[split])
    lo, hi = actual.min() * 0.85, actual.max() * 1.15

    # Perfect-prediction reference line.
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines",
                             line=dict(color="#111", dash="dash", width=1),
                             name="perfect prediction", hoverinfo="skip"))

    for m in selected:
        pred = dollars(PRED[m][split])
        err = (pred - actual) / actual * 100
        fig.add_trace(go.Scatter(
            x=actual, y=pred, mode="markers", name=LABELS[m],
            marker=dict(size=6, opacity=0.65, color=PALETTE[m]),
            customdata=err,
            hovertemplate="actual $%{x:,.0f}<br>predicted $%{y:,.0f}"
                          "<br>error %{customdata:.1f}%<extra></extra>"))

    if log_scale:
        fig.update_xaxes(type="log"); fig.update_yaxes(type="log")
    return style(fig, f"Predicted vs actual - {split} set",
                 "Actual sale price ($)", "Predicted sale price ($)")


def metrics_table(split):
    rows = []
    for m in MODELS:
        d = RESULTS["metrics"][m][split]
        rows.append({
            "Model": LABELS[m],
            "RMSE (log)": round(d["rmse_log"], 4),
            "R²": round(d["r2"], 3),
            "Median APE": f"{d['med_ape']:.2f}%",
            "Within 10%": f"{d['within_10pct']:.1f}%",
            "RMSE ($)": f"${d['rmse_dollars']:,.0f}",
            "Fit time (s)": round(RESULTS["timings"].get(m, 0), 3),
        })
    return pd.DataFrame(rows)


def agreement_note():
    ref = PARAMS["sklearn"]
    lines = ["**Do the three implementations agree?**", ""]
    for m in ["manual_torch", "standard_torch"]:
        d = np.abs(PARAMS[m] - ref)
        lines.append(f"- {LABELS[m]} vs scikit-learn: max coefficient "
                     f"difference **{d.max():.6f}**, correlation "
                     f"**{np.corrcoef(PARAMS[m], ref)[0,1]:.6f}**")
    c = RESULTS["config"]
    lines += ["", f"All three solve the same ridge problem (alpha={c['alpha']:.0f}, "
              f"weight_decay={c['weight_decay']:.4f} = 2·alpha/n). scikit-learn "
              f"solves it in closed form; the two PyTorch versions reach the same "
              f"optimum by gradient descent over {c['epochs']} epochs at lr={c['lr']}."]
    return "\n".join(lines)


def loss_curves():
    fig = go.Figure()
    for m in ["manual_torch", "standard_torch"]:
        h = np.array(RESULTS["history"][m])
        fig.add_trace(go.Scatter(x=h[:, 0], y=h[:, 1], mode="lines",
                                 name=LABELS[m], line=dict(color=PALETTE[m], width=2)))
    sk_mse = RESULTS["metrics"]["sklearn"]["train"]["rmse_log"] ** 2
    fig.add_hline(y=sk_mse, line_dash="dash", line_color="#2563eb",
                  annotation_text="scikit-learn exact optimum")
    fig.update_yaxes(type="log")
    return style(fig, "Gradient descent converging to the closed-form solution",
                 "Epoch", "Training MSE (log scale)")

## Tab 2 — Distributions

The log toggle makes the central modelling decision visible: raw price is skewed, log price is symmetric. Train vs test overlays show whether the chronological split caught any real drift.

In [ ]:
# -------------------------------------------------------- TAB 2: distributions

NUMERIC_FEATURES = sorted(
    c for c in RAW_TRAIN.columns
    if pd.api.types.is_numeric_dtype(RAW_TRAIN[c])
    and c not in ("SalePrice", "Yr Sold") and RAW_TRAIN[c].nunique() > 2
)


def target_distribution(use_log):
    fig = go.Figure()
    for name, frame, colour in [("Train (2006-08)", RAW_TRAIN, "#2563eb"),
                                ("Test (2010)", RAW_TEST, "#d97706")]:
        v = np.log(frame.SalePrice) if use_log else frame.SalePrice
        fig.add_trace(go.Histogram(x=v, name=name, opacity=0.6,
                                   marker_color=colour, nbinsx=45,
                                   histnorm="probability density"))
    fig.update_layout(barmode="overlay")
    return style(fig,
                 "Sale price: raw is right-skewed, log is symmetric"
                 if not use_log else "Sale price in log space",
                 "log(sale price)" if use_log else "Sale price ($)", "Density")


def feature_distribution(col):
    fig = go.Figure()
    for name, frame, colour in [("Train (2006-08)", RAW_TRAIN, "#2563eb"),
                                ("Test (2010)", RAW_TEST, "#d97706")]:
        fig.add_trace(go.Histogram(x=frame[col], name=name, opacity=0.6,
                                   marker_color=colour, nbinsx=40,
                                   histnorm="probability density"))
    fig.update_layout(barmode="overlay")
    return style(fig, f"{col} - train vs test", col, "Density")


def feature_vs_price(col):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=RAW_TRAIN[col], y=RAW_TRAIN.SalePrice, mode="markers",
        marker=dict(size=5, opacity=0.4, color="#2563eb"), name="Train",
        hovertemplate=f"{col}: %{{x}}<br>price $%{{y:,.0f}}<extra></extra>"))
    fig.update_yaxes(type="log")
    r = RAW_TRAIN[[col, "SalePrice"]].corr().iloc[0, 1]
    return style(fig, f"{col} vs sale price (correlation {r:+.2f})",
                 col, "Sale price ($, log axis)")


def shift_note(col):
    a, b = RAW_TRAIN[col], RAW_TEST[col]
    return (f"**{col}** — train mean `{a.mean():,.1f}`, test mean `{b.mean():,.1f}` "
            f"(shift of `{(b.mean()-a.mean())/a.std():+.2f}` train standard deviations). "
            f"Train median price `${RAW_TRAIN.SalePrice.median():,.0f}` vs test "
            f"`${RAW_TEST.SalePrice.median():,.0f}`.")

## Tab 3 — Appraisal policy

Regression has no probability threshold, so the decision variable is an **error tolerance**. The automation rule is built from the **2009 validation set** — the lender cannot know a house's error in advance, so confidence comes from how the model behaved on similar past houses.

In [ ]:
# ----------------------------------------------------- TAB 3: appraisal policy

# Which houses can the lender auto-value? It cannot know the error in advance,
# so confidence must come from data it HAS seen: the 2009 validation set.
# We bucket validation houses by predicted price, measure the error the model
# actually made in each bucket, and trust a 2010 house only if its bucket was
# reliable on validation. No test data is used to build the policy.
_val_pred = dollars(PRED["sklearn"]["val"])
_val_true = dollars(TRUTH["val"])
_edges = np.quantile(_val_pred, np.linspace(0, 1, 11))
_edges[0], _edges[-1] = 0, np.inf
_val_bucket = np.clip(np.digitize(_val_pred, _edges) - 1, 0, 9)
_val_ape = np.abs(_val_pred - _val_true) / _val_true * 100


def bucket_risk(confidence):
    """Error level the model stayed under for `confidence`% of validation
    houses in each price bucket. Higher confidence = stricter bar."""
    return np.array([np.percentile(_val_ape[_val_bucket == b], confidence)
                     for b in range(10)])


def policy_outcomes(tolerance, loss_rate, confidence=75):
    pred = dollars(PRED["sklearn"]["test"])
    true = dollars(TRUTH["test"])
    bucket = np.clip(np.digitize(pred, _edges) - 1, 0, 9)

    auto = bucket_risk(confidence)[bucket] <= tolerance          # the decision, made blind
    ape = np.abs(pred - true) / true * 100
    good = ape <= tolerance                           # what actually happened

    over = (pred > true) & auto & ~good
    under = (pred < true) & auto & ~good

    # Asymmetric costs, per the business framing.
    cost_over = (pred[over] - true[over]).sum() * loss_rate
    cost_under = under.sum() * 1000
    cost_escalated = (~auto).sum() * APPRAISAL_COST
    total = cost_over + cost_under + cost_escalated
    baseline = len(true) * APPRAISAL_COST

    return dict(n=len(true), auto=auto, good=good, over=over, under=under,
                cost_over=cost_over, cost_under=cost_under,
                cost_escalated=cost_escalated, total=total, baseline=baseline)


def outcome_matrix(tolerance, loss_rate, confidence=75):
    o = policy_outcomes(tolerance, loss_rate, confidence)
    a, g = o["auto"], o["good"]
    cells = np.array([[int((a & g).sum()), int((a & ~g).sum())],
                      [int((~a & g).sum()), int((~a & ~g).sum())]])
    text = [[f"<b>{cells[0,0]}</b><br>correct automation<br>$0", 
             f"<b>{cells[0,1]}</b><br>BAD VALUATION<br>loss incurred"],
            [f"<b>{cells[1,0]}</b><br>needless appraisal<br>$500 each",
             f"<b>{cells[1,1]}</b><br>correct escalation<br>$500 well spent"]]
    fig = go.Figure(go.Heatmap(
        z=cells, text=text, texttemplate="%{text}", showscale=False,
        x=["Actually within tolerance", "Actually outside tolerance"],
        y=["Escalated to appraiser", "Auto-valued"],
        colorscale=[[0, "#f1f5f9"], [1, "#bfdbfe"]], hoverinfo="skip"))
    return style(fig, f"Outcomes at ±{tolerance:.0f}% tolerance "
                      f"({o['n']} houses, 2010)", "", "")


def policy_summary(tolerance, loss_rate, confidence=75):
    o = policy_outcomes(tolerance, loss_rate, confidence)
    auto_pct = o["auto"].mean() * 100
    bad = int((o["auto"] & ~o["good"]).sum())
    saving = o["baseline"] - o["total"]
    return (
        f"### Policy at ±{tolerance:.0f}% tolerance\n\n"
        f"| | |\n|---|---|\n"
        f"| Auto-valued | **{auto_pct:.1f}%** ({int(o['auto'].sum())} of {o['n']}) |\n"
        f"| Sent to an appraiser | {100-auto_pct:.1f}% ({int((~o['auto']).sum())}) |\n"
        f"| Bad valuations that slipped through | **{bad}** |\n\n"
        f"**Cost on 2010 volume**\n\n"
        f"| Component | Cost |\n|---|---|\n"
        f"| Appraisals ({int((~o['auto']).sum())} × $500) | ${o['cost_escalated']:,.0f} |\n"
        f"| Over-valuation losses ({int(o['over'].sum())} houses @ {loss_rate:.0%}) "
        f"| ${o['cost_over']:,.0f} |\n"
        f"| Lost deals from under-valuation ({int(o['under'].sum())} × $1,000) "
        f"| ${o['cost_under']:,.0f} |\n"
        f"| **Total** | **${o['total']:,.0f}** |\n"
        f"| All-human baseline ({o['n']} × $500) | ${o['baseline']:,.0f} |\n"
        f"| **Net saving** | **${saving:,.0f}** |\n")


def cost_curve(loss_rate, confidence=75):
    tols = np.arange(2, 31, 1.0)
    totals = [policy_outcomes(t, loss_rate, confidence)["total"] for t in tols]
    best = tols[int(np.argmin(totals))]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=tols, y=totals, mode="lines+markers",
                             line=dict(color="#2563eb", width=2), name="Total cost"))
    fig.add_hline(y=len(TRUTH["test"]) * APPRAISAL_COST, line_dash="dash",
                  line_color="#94a3b8", annotation_text="all-human baseline")
    fig.add_vline(x=best, line_dash="dot", line_color="#16a34a",
                  annotation_text=f"cheapest: ±{best:.0f}%")
    return style(fig, "Total cost across tolerance settings",
                 "Tolerance (± % error)", "Total cost ($)")

## Tab 4 — Value a house

One 2010 property scored by all four models against its true sale price.

In [ ]:
# ------------------------------------------------------- TAB 4: value a house

HOUSE_CHOICES = [
    (f"#{i} — {int(r['Gr Liv Area']):,} sqft, {r['Neighborhood']}, "
     f"built {int(r['Year Built'])}, quality {int(r['Overall Qual'])}/10", i)
    for i, r in RAW_TEST.iterrows()
]


def value_house(idx):
    row = RAW_TEST.iloc[[int(idx)]]
    true = float(row.SalePrice.iloc[0])

    fig = go.Figure()
    lines = [f"### Actual sale price: **${true:,.0f}**", "",
             "| Model | Predicted | Error |", "|---|---|---|"]
    for m in MODELS:
        pred = float(dollars(predict_rows(m, row))[0])
        err = (pred - true) / true * 100
        lines.append(f"| {LABELS[m]} | ${pred:,.0f} | {err:+.1f}% |")
        fig.add_trace(go.Bar(x=[LABELS[m]], y=[pred], marker_color=PALETTE[m],
                             name=LABELS[m], showlegend=False,
                             text=f"${pred:,.0f}", textposition="outside"))
    fig.add_hline(y=true, line_dash="dash", line_color="#111",
                  annotation_text=f"actual ${true:,.0f}")

    r = row.iloc[0]
    lines += ["", "**This property**", "",
              f"- {int(r['Gr Liv Area']):,} sqft living area, "
              f"{int(r['Total SF']):,} sqft total",
              f"- {r['Neighborhood']}, built {int(r['Year Built'])}, "
              f"{int(r['Age'])} years old at sale",
              f"- Overall quality {int(r['Overall Qual'])}/10, "
              f"{r['Total Baths']} baths"]
    return style(fig, "What each model would have told the lender",
                 "", "Predicted price ($)"), "\n".join(lines)

## The page

`gr.Blocks` is the layout; `.change()` wires a component to a function. `demo.launch()` uses Gradio's defaults — `127.0.0.1:7860` — which is exactly what CI curls.

In [ ]:
# ------------------------------------------------------------------ the page

INTRO = """
# Ames automated valuation model
**Louis de Perier — DAIIST Assignment 1**

A mortgage lender prices properties offered as collateral. A human appraisal
costs **$500** and takes 5–10 days; this model prices the easy cases instantly
so appraisers are reserved for the hard ones. Trained on **2006–2008** sales,
tuned on **2009**, and evaluated on **301 unseen 2010 sales**.

*All figures below are loaded from artifacts written by `main.py train`.
Nothing is trained here.*
"""

with gr.Blocks(title="Ames AVM") as demo:
    gr.Markdown(INTRO)

    # ---- TAB 1 -------------------------------------------------------------
    with gr.Tab("Model comparison"):
        with gr.Row():
            pick = gr.CheckboxGroup(MODELS, value=["sklearn", "manual_torch",
                                                   "standard_torch"],
                                    label="Models to plot")
            split = gr.Radio(["test", "val", "train"], value="test", label="Split")
            logax = gr.Checkbox(value=True, label="Log axes")
        plot1 = gr.Plot(scatter_pred_actual(["sklearn", "manual_torch",
                                             "standard_torch"], "test", True))
        table1 = gr.Dataframe(metrics_table("test"), label="Metrics")
        gr.Markdown(agreement_note())
        gr.Plot(loss_curves())

        def refresh(models, sp, lg):
            return scatter_pred_actual(models, sp, lg), metrics_table(sp)
        for ctrl in (pick, split, logax):
            ctrl.change(refresh, [pick, split, logax], [plot1, table1])

    # ---- TAB 2 -------------------------------------------------------------
    with gr.Tab("Distributions"):
        tlog = gr.Checkbox(value=False, label="Show target in log space")
        plot2 = gr.Plot(target_distribution(False))
        tlog.change(target_distribution, tlog, plot2)

        feat = gr.Dropdown(NUMERIC_FEATURES, value="Total SF", label="Feature")
        note2 = gr.Markdown(shift_note("Total SF"))
        with gr.Row():
            plot3 = gr.Plot(feature_distribution("Total SF"))
            plot4 = gr.Plot(feature_vs_price("Total SF"))
        feat.change(lambda c: (feature_distribution(c), feature_vs_price(c),
                               shift_note(c)), feat, [plot3, plot4, note2])

    # ---- TAB 3 -------------------------------------------------------------
    with gr.Tab("Appraisal policy"):
        gr.Markdown(
            "Regression has no probability threshold, so the lender's decision "
            "variable is an **error tolerance**. A price band is auto-valued only "
            "if the model stayed within tolerance for the required share of "
            "**2009 validation** houses in that band — the test set is never used "
            "to build the policy.")
        with gr.Row():
            tol = gr.Slider(2, 30, value=10, step=1, label="Tolerance (± % error)")
            conf = gr.Slider(50, 95, value=75, step=5,
                             label="Confidence required (% of validation houses)")
            loss = gr.Slider(0.05, 0.50, value=0.25, step=0.05,
                             label="Loss on over-valuation (share of excess)")
        with gr.Row():
            plot5 = gr.Plot(outcome_matrix(10, 0.25, 75))
            summ = gr.Markdown(policy_summary(10, 0.25, 75))
        plot6 = gr.Plot(cost_curve(0.25, 75))

        def repolicy(t, c, l):
            return outcome_matrix(t, l, c), policy_summary(t, l, c), cost_curve(l, c)
        for ctrl in (tol, conf, loss):
            ctrl.change(repolicy, [tol, conf, loss], [plot5, summ, plot6])

    # ---- TAB 4 -------------------------------------------------------------
    with gr.Tab("Value a house"):
        house = gr.Dropdown(HOUSE_CHOICES, value=0, label="2010 property")
        with gr.Row():
            plot7 = gr.Plot(value_house(0)[0])
            det = gr.Markdown(value_house(0)[1])
        house.change(value_house, house, [plot7, det])


if __name__ == "__main__":
    demo.launch(theme=gr.themes.Soft())